### Exception Handling in Python :Your Complete Guide to Writing Programs That Never (Unexpectedly) Crash

---

Picture a tightrope walker crossing between two skyscrapers. Without a safety net below, one small slip means disaster. With a safety net? They can still fall, but they're caught so they recover and try again.

**Exception handling is your code's safety net.**

No matter how carefully you write code, the real world is unpredictable:
- A user types `"five"` when you expected `5`
- A file gets deleted while your program is running
- The internet goes down mid-request
- A database times out

Without exception handling, your program **crashes**. With it, your program **recovers gracefully**.

**What you'll master in this notebook:**
1. The three types of errors in Python
2. The difference between Errors and Exceptions
3. Python's full exception hierarchy
4. `try`, `except`, `else`, `finally` - the complete set
5. Raising exceptions with `raise`
6. Re-raising and exception chaining
7. The `assert` statement
8. Creating your own custom exceptions
9. Logging exceptions like a professional
10. Advanced tools: `contextlib.suppress`, `sys.exc_info()`
11. Best practices used in real production systems


### **1. The Three Types of Errors in Programming**

Before we handle exceptions, let's understand what can go wrong. There are three broad categories of errors in any programming language.

#### Type 1: Syntax Errors (a.k.a. "You broke the grammar rules")

A **Syntax Error** happens when Python can't even understand what you wrote. It's like sending a grammatically broken sentence which the reader can't make sense of.

These are caught **before** the program even runs. Python's parser reads your code and immediately says "I don't understand this."

**Common causes:**
- Missing colons after `if`, `for`, `def`, `class`
- Mismatched brackets `(`, `)`, `[`, `]`, `{`, `}`
- Misspelled keywords
- Bad indentation (`IndentationError` is a subtype of `SyntaxError`)

In [80]:
# Missing colon after 'if'
if True
  print("Hello")

SyntaxError: expected ':' (1621482940.py, line 2)

In [82]:
# Mismatched brackets
result = (1 + 2

SyntaxError: incomplete input (1906561714.py, line 2)

In [86]:
# Bad indentation (IndentationError)
def greet():
print("Hi")   # not indented

IndentationError: expected an indented block after function definition on line 2 (3583019299.py, line 3)

In [88]:
print("Syntax errors are caught before the program runs.")
print("Python reads the whole file first, then reports all syntax issues.")
print("You will never see output from a file with a syntax error.")

Syntax errors are caught before the program runs.
Python reads the whole file first, then reports all syntax issues.
You will never see output from a file with a syntax error.


#### Type 2: Logical Errors (a.k.a. "You said the wrong thing")

A **Logical Error** is the trickiest kind. The code runs perfectly, no crashes but it produces the **wrong answer**. Python has no way to detect these. Only you (or tests) can catch them.

These are the bugs that keep senior engineers up at night!

**Common causes:**
- Using `>` instead of `>=`
- Off-by-one errors in loops
- Wrong formula
- Misunderstanding the requirements

In [91]:
# Logical error: no crash, but wrong result!
def calculate_average(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / 10  # BUG: hardcoded 10 instead of len(numbers)!

scores = [80, 90, 75, 95, 85]
avg = calculate_average(scores)
print(f"Average score: {avg}")           
print(f"Correct average: {sum(scores) / len(scores)}") 

Average score: 42.5
Correct average: 85.0


#### Type 3: Runtime Errors / Exceptions (a.k.a. "Something went wrong while running")

**Runtime errors** (also called **exceptions**) occur while the program is actually executing. The code has correct syntax and correct logic but something unexpected happens during execution. These are what exception handling is all about.

**Common causes:**
- Dividing by zero
- Accessing a list index that doesn't exist
- Opening a file that doesn't exist
- Calling a method on the wrong type
- Network failures, memory limits, etc.

In [94]:
# All of these cause runtime errors (exceptions)
# Each line below would crash the program — run them one at a time

examples = [
    ("ZeroDivisionError",   lambda: 1 / 0),
    ("IndexError",          lambda: [1, 2, 3][10]),
    ("KeyError",            lambda: {"a": 1}["b"]),
    ("TypeError",           lambda: "5" + 5),
    ("ValueError",          lambda: int("hello")),
    ("AttributeError",      lambda: "text".non_existent_method()),
    ("NameError",           lambda: undefined_variable),
    ("FileNotFoundError",   lambda: open("ghost.txt")),
]

print("Demonstrating common runtime exceptions:")

for name, trigger in examples:
    try:
        trigger()
    except Exception as e:
        print(f"  {name:<22}: {e}")


Demonstrating common runtime exceptions:
  ZeroDivisionError     : division by zero
  IndexError            : list index out of range
  KeyError              : 'b'
  TypeError             : can only concatenate str (not "int") to str
  ValueError            : invalid literal for int() with base 10: 'hello'
  AttributeError        : 'str' object has no attribute 'non_existent_method'
  NameError             : name 'undefined_variable' is not defined
  FileNotFoundError     : [Errno 2] No such file or directory: 'ghost.txt'


### **2. Errors vs Exceptions**

You'll often hear these two words used interchangeably, but they have subtle differences in Python:

| | **Errors** | **Exceptions** |
|---|---|---|
| **When detected** | Before execution (compile/parse time) | During execution (runtime) |
| **Examples** | `SyntaxError`, `IndentationError` | `ValueError`, `ZeroDivisionError`, `FileNotFoundError` |
| **Handleable?** | Usually Not (program can't start) | Yes, this is what `try/except` is for |
| **Python's action** | Stops before running | Creates an exception object and raises it |

### The Exception Object

When a runtime exception occurs, Python automatically creates an **exception object** containing:
- The **type** of exception (e.g., `ValueError`)
- A **message** describing what went wrong
- A **traceback** showing exactly where in the code it happened

If you don't handle it, Python prints the traceback and your program stops. If you do handle it, you decide what happens next.

### **3. Python's Exception Hierarchy**

Python's exceptions form a **family tree**. Every exception is a class, and they inherit from each other. Understanding this tree helps you catch the right exceptions.

```
BaseException
├── SystemExit              ← raised by sys.exit()
├── KeyboardInterrupt       ← raised by Ctrl+C
├── GeneratorExit           ← raised when a generator closes
└── Exception               ← parent of ALL regular exceptions
    ├── ArithmeticError
    │   ├── ZeroDivisionError
    │   ├── OverflowError
    │   └── FloatingPointError
    ├── LookupError
    │   ├── IndexError
    │   └── KeyError
    ├── OSError  (same as IOError, EnvironmentError)
    │   ├── FileNotFoundError
    │   ├── PermissionError
    │   ├── FileExistsError
    │   └── TimeoutError
    ├── ValueError
    ├── TypeError
    ├── NameError
    │   └── UnboundLocalError
    ├── AttributeError
    ├── ImportError
    │   └── ModuleNotFoundError
    ├── RuntimeError
    │   └── RecursionError
    ├── StopIteration
    ├── MemoryError
    └── AssertionError
```

**Key insight:** If you catch a parent exception, you catch its children exceptions too. Catching `OSError` catches `FileNotFoundError`, `PermissionError`, and all other OS-related errors.


In [102]:
# Demonstrating the parent-child relationship

# Catching a parent catches all its children
try:
    result = 1 / 0          # ZeroDivisionError
except ArithmeticError as e:
    # ZeroDivisionError is a child of ArithmeticError
    print(f"Caught as ArithmeticError: {type(e).__name__}: {e}")

try:
    x = [1, 2, 3][99]      # IndexError
except LookupError as e:
    # IndexError is a child of LookupError
    print(f"Caught as LookupError: {type(e).__name__}: {e}")

try:
    open("missing.txt")    # FileNotFoundError
except OSError as e:
    # FileNotFoundError is a child of OSError
    print(f"Caught as OSError: {type(e).__name__}: {e}")

print()
# Check the hierarchy with isinstance
e = ValueError("test")
print(f"ValueError is an Exception : {isinstance(e, Exception)}")
print(f"ValueError is a BaseException: {isinstance(e, BaseException)}")
print(f"ValueError is a TypeError  : {isinstance(e, TypeError)}")

Caught as ArithmeticError: ZeroDivisionError: division by zero
Caught as LookupError: IndexError: list index out of range
Caught as OSError: FileNotFoundError: [Errno 2] No such file or directory: 'missing.txt'

ValueError is an Exception : True
ValueError is a BaseException: True
ValueError is a TypeError  : False


### **4. The Most Important Exceptions**

Before learning to handle exceptions, know what you're handling. Here are the most common ones you'll encounter in real projects:

In [107]:
import traceback

def demo(description, expression):
    try:
        eval(expression)
    except Exception as e:
        print(f"[{type(e).__name__}]")
        print(f"  Scenario : {description}")
        print(f"  Message  : {e}")
        print()

# Set up a namespace for eval
import builtins
print()

# TypeError
try:
    result = "age: " + 25
except TypeError as e:
    print(f"[TypeError]")
    print(f"  Scenario : Adding a string and an integer")
    print(f"  Message  : {e}")
    print(f"  Fix      : Convert types — 'age: ' + str(25)  or  f'age: {{25}}'")
    print()

# ValueError
try:
    num = int("hello")
except ValueError as e:
    print(f"[ValueError]")
    print(f"  Scenario : int() called with non-numeric string")
    print(f"  Message  : {e}")
    print(f"  Fix      : Validate input before converting")
    print()

# IndexError
try:
    items = [10, 20, 30]
    print(items[5])
except IndexError as e:
    print(f"[IndexError]")
    print(f"  Scenario : Accessing index 5 in a 3-item list")
    print(f"  Message  : {e}")
    print(f"  Fix      : Check len(list) first, or use .get() pattern")
    print()

# KeyError
try:
    user = {"name": "Alice", "age": 28}
    print(user["email"])
except KeyError as e:
    print(f"[KeyError]")
    print(f"  Scenario : Accessing key 'email' that doesn't exist in dict")
    print(f"  Message  : {e}")
    print(f"  Fix      : Use dict.get('email', default) instead")
    print()

# AttributeError
try:
    x = 42
    x.upper()
except AttributeError as e:
    print(f"[AttributeError]")
    print(f"  Scenario : Calling .upper() on an integer (not a string)")
    print(f"  Message  : {e}")
    print(f"  Fix      : Check the type before calling the method")
    print()

# ZeroDivisionError
try:
    result = 100 / 0
except ZeroDivisionError as e:
    print(f"[ZeroDivisionError]")
    print(f"  Scenario : Dividing by zero")
    print(f"  Message  : {e}")
    print(f"  Fix      : Check denominator != 0 before dividing")
    print()

# NameError
try:
    print(undefined_variable)
except NameError as e:
    print(f"[NameError]")
    print(f"  Scenario : Using a variable that was never defined")
    print(f"  Message  : {e}")
    print(f"  Fix      : Check spelling, make sure the variable is assigned first")
    print()

# RecursionError
def infinite_recursion():
    return infinite_recursion()

try:
    infinite_recursion()
except RecursionError as e:
    print(f"[RecursionError]")
    print(f"  Scenario : A function calling itself endlessly")
    print(f"  Message  : {str(e)[:50]}")
    print(f"  Fix      : Add a base case to stop the recursion")



[TypeError]
  Scenario : Adding a string and an integer
  Message  : can only concatenate str (not "int") to str
  Fix      : Convert types — 'age: ' + str(25)  or  f'age: {25}'

[ValueError]
  Scenario : int() called with non-numeric string
  Message  : invalid literal for int() with base 10: 'hello'
  Fix      : Validate input before converting

[IndexError]
  Scenario : Accessing index 5 in a 3-item list
  Message  : list index out of range
  Fix      : Check len(list) first, or use .get() pattern

[KeyError]
  Scenario : Accessing key 'email' that doesn't exist in dict
  Message  : 'email'
  Fix      : Use dict.get('email', default) instead

[AttributeError]
  Scenario : Calling .upper() on an integer (not a string)
  Message  : 'int' object has no attribute 'upper'
  Fix      : Check the type before calling the method

[ZeroDivisionError]
  Scenario : Dividing by zero
  Message  : division by zero
  Fix      : Check denominator != 0 before dividing

[NameError]
  Scenario : Using

### **5. The `try` / `except` Block**

The `try/except` block is the foundation of exception handling in Python.

```python
try:
    # Code that might fail
except SomeException:
    # What to do if it fails
```

**How it works:**
1. Python runs the `try` block line by line
2. If an exception occurs, Python **immediately jumps** to the connecting `except` block
3. Lines after the exception in `try` are **skipped**
4. After the `except` block runs, the program **continues normally**

**The golden rule:** Be as **specific** as possible about which exception you catch.


In [112]:
# Basic try/except
try:
    result = 10 / 2
    print(f"Success: 10 / 2 = {result}")
except ZeroDivisionError:
    print("Cannot divide by zero!")

print("Program continues here...\n")

# When an exception is triggered
try:
    print("Line 1: About to divide...")
    result = 10 / 0             # Exception happens HERE
    print("Line 2: This line is skipped!")  # Never runs
    print("Line 3: Also skipped!")          # Never runs
except ZeroDivisionError:
    print("Caught: ZeroDivisionError!")  #execution flow jumps here

print("Program continues here...")

Success: 10 / 2 = 5.0
Program continues here...

Line 1: About to divide...
Caught: ZeroDivisionError!
Program continues here...


In [114]:
# Capturing the exception object with 'as'
# The exception object contains the error message and more

try:
    data = {"price": "19.99", "currency": "USD"}
    quantity = int(data["quantity"])    # KeyError: 'quantity' not in dict
except KeyError as e:
    # 'e' is the exception object — access the message with str(e) or e.args
    print(f"KeyError caught!")
    print(f"  Exception type    : {type(e).__name__}")
    print(f"  Exception message : {e}")
    print(f"  Exception args    : {e.args}")

print()

try:
    age = int("twenty five")
except ValueError as e:
    print(f"ValueError caught!")
    print(f"  Full error: {e}")
    # You can use the exception info to make smart decisions
    if "invalid literal" in str(e):
        print("  Hint: The input contains letters, not digits")


KeyError caught!
  Exception type    : KeyError
  Exception message : 'quantity'
  Exception args    : ('quantity',)

ValueError caught!
  Full error: invalid literal for int() with base 10: 'twenty five'
  Hint: The input contains letters, not digits


### Real-World Scenario: Payment Processing Like Stripe

Stripe processes millions of payments per day. Every payment attempt can fail for dozens of different reasons — invalid card number, insufficient funds, expired card, network timeout. Each failure needs a different response:


In [116]:
# Simulated payment processing system like Stripe or PayPal

class CardExpiredError(Exception): 
    pass

class InsufficientFundsError(Exception): 
    pass

class InvalidCardError(Exception): 
    pass

class NetworkTimeoutError(Exception): 
    pass

def process_payment(card_number, amount, card_balance):
    """Process a payment which can fail in many ways."""
    if not card_number.isdigit() or len(card_number) != 16:
        raise InvalidCardError(f"Card '{card_number}' is not a valid 16-digit number")
    if card_number.startswith("0000"):
        raise CardExpiredError("Card has expired")
    if card_balance < amount:
        raise InsufficientFundsError(
            f"Card balance ${card_balance:.2f} is less than charge ${amount:.2f}"
        )
    return {"status": "success", "charged": amount, "remaining": card_balance - amount}

def checkout(card_number, amount, card_balance):
    """Checkout with friendly error messages, just like a real payment page."""
    print(f"  Processing ${amount:.2f} on card ending in {card_number[-4:]}...")
    try:
        result = process_payment(card_number, amount, card_balance)
        print(f"  Payment approved! Remaining balance: ${result['remaining']:.2f}")
    except InvalidCardError as e:
        print(f"  Declined: Invalid card details. ({e})")
        print(f"  Action: Ask user to re-enter card number")
    except CardExpiredError:
        print(f"  Declined: Your card has expired.")
        print(f"  Action: Ask user to update their payment method")
    except InsufficientFundsError as e:
        print(f"  Declined: Insufficient funds.")
        print(f"  Action: Suggest a smaller order or different payment method")
    except NetworkTimeoutError:
        print(f"  Error: Network timeout. Please try again.")
        print(f"  Action: Retry automatically up to 3 times")
    print()

print("Stripe-style Payment Processing\n")
checkout("4532015112830366", 49.99, 200.00)   # Success
checkout("abc123",           49.99, 200.00)   # Invalid card
checkout("0000111122223333", 49.99, 200.00)   # Expired
checkout("4532015112830366", 49.99, 30.00)    # Insufficient funds


Stripe-style Payment Processing

  Processing $49.99 on card ending in 0366...
  Payment approved! Remaining balance: $150.01

  Processing $49.99 on card ending in c123...
  Declined: Invalid card details. (Card 'abc123' is not a valid 16-digit number)
  Action: Ask user to re-enter card number

  Processing $49.99 on card ending in 3333...
  Declined: Your card has expired.
  Action: Ask user to update their payment method

  Processing $49.99 on card ending in 0366...
  Declined: Insufficient funds.
  Action: Suggest a smaller order or different payment method



### **6. The `else` Clause : "Only if Everything Went Fine"**

The `else` block in a `try` statement runs **only if no exception occurred** in the `try` block. It's the "success path."

```python
try:
    risky_operation()
except SomeError:
    handle_error()
else:
    # Only runs if try block succeeded with no exceptions
    do_success_work()
```

**Why use `else` instead of putting success code in `try`?**

Because code in `try` should be only the **risky part**. If your success code also raises an exception, you don't want it accidentally caught by your `except` block. `else` keeps the risk zone tight.

In [119]:
# Without else — risky! Success code is inside try
def divide_v1(a, b):
    try:
        result = a / b
        # This is 'success code' but it's inside try
        # If this line had an error, the except below would catch it!
        formatted = f"Result: {result:.4f}"
        print(formatted)
    except ZeroDivisionError:
        print("Cannot divide by zero")

# With else — cleaner and safer
def divide_v2(a, b):
    try:
        result = a / b          # ONLY the risky part is in try
    except ZeroDivisionError:
        print("Cannot divide by zero")
    else:
        # Only runs on success
        formatted = f"Result: {result:.4f}"
        print(formatted)

print("divide_v1 (without else):")
divide_v1(10, 3)
divide_v1(10, 0)

print("\ndivide_v2 (with else, a better style):")
divide_v2(10, 3)
divide_v2(10, 0)

divide_v1 (without else):
Result: 3.3333
Cannot divide by zero

divide_v2 (with else, a better style):
Result: 3.3333
Cannot divide by zero


### **7. The `finally` Clause : "Always Run This, No Matter What"**

The `finally` block **always runs**, whether the `try` succeeded, whether an exception was caught, whether an exception was NOT caught, even if there's a `return` statement.

```python
try:
    risky_operation()
except SomeError:
    handle_error()
finally:
    # ALWAYS runs no matter what happened above
    cleanup()
```

**What is `finally` used for?**

Cleanup operations that must happen regardless of success or failure:
- Closing database connections
- Closing files
- Releasing network sockets
- Releasing locks (so other threads aren't blocked)
- Logging "operation finished"

**Real-world analogy:** A surgeon must close the patient up (suture the incision) whether the operation succeeded or failed. `finally` is that suturing step.

In [122]:
# finally always runs, even with an exception
print("Scenario 1: No exception")
try:
    print("try: doing work")
    result = 10 / 2
except ZeroDivisionError:
    print("except: caught error")
else:
    print(f"else: success! result = {result}")
finally:
    print("  finally: always runs")

print()

print("Scenario 2: Exception caught")
try:
    print("try: doing work")
    result = 10 / 0
except ZeroDivisionError:
    print("except: caught ZeroDivisionError")
finally:
    print("finally: always runs!")

Scenario 1: No exception
try: doing work
else: success! result = 5.0
  finally: always runs

Scenario 2: Exception caught
try: doing work
except: caught ZeroDivisionError
finally: always runs!


### Real-World Scenario: Database Connections Like PostgreSQL / MySQL

Every database operation in production must close its connection — even if an error occurs. Leaving connections open wastes server resources and can crash apps under high load. `finally` is the industry standard for this:


In [ ]:
# Simulated database connection management like SQLAlchemy

class DatabaseConnection:
    """Simulates a database connection (like PostgreSQL, MySQL)."""
    
    def __init__(self, db_name):
        self.db_name = db_name
        self.is_open = False
    
    def connect(self):
        self.is_open = True
        print(f"[DB] Connected to '{self.db_name}'")
    
    def execute(self, query):
        if not self.is_open:
            raise RuntimeError("Cannot execute: connection is closed!")
        if "DROP" in query.upper():
            raise PermissionError("DROP operations require admin privileges!")
        print(f"[DB] Executed: {query}")
        return [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]
    
    def close(self):
        self.is_open = False
        print(f"[DB] Connection to '{self.db_name}' closed")

def run_query(query):
    db = DatabaseConnection("production_db")
    try:
        db.connect()
        results = db.execute(query)
        print(f"[DB] Got {len(results)} rows")
        return results
    except PermissionError as e:
        print(f"[ERROR] Permission denied: {e}")
        return []
    except Exception as e:
        print(f"[ERROR] Unexpected error: {e}")
        return []
    finally:
        db.close()   # ALWAYS close the connection!
        print(f"[DB] Resources freed\n")

print("Query 1: Safe SELECT")
run_query("SELECT * FROM users WHERE active=1")

print("Query 2: Dangerous DROP (permission error)")
run_query("DROP TABLE users")

### **8. The Complete Structure: `try` / `except` / `else` / `finally`**

All four clauses together form the complete exception handling toolkit. Here's when each runs:

```
try:
    [risky code]
except SomeError:
    [runs only if 'try' raises SomeError]
except AnotherError:
    [runs only if 'try' raises AnotherError]
else:
    [runs only if 'try' had no exceptions]
finally:
    [always runs success or failure, caught or uncaught]
```

**Memory trick — think of it as a story:**
- `try` : "Let me try this"
- `except` : "If it goes wrong like THIS, I'll do THAT"
- `else` : "If it went perfectly, here's what I'll do next"
- `finally` : "Either way, I'll always clean up."

In [126]:
def safe_divide(a, b):
    try:
        result = a / b                         # risky operation
    except ZeroDivisionError:
        print("Error: Cannot divide by zero.")
    except TypeError as e:
        print(f"TypeError: {e}")
    else:
        print(f"Result: {result}")           # runs only if no exception
    finally:
        print("Execution complete.\n")      # always runs

safe_divide(10, 2)    # Normal case
safe_divide(10, 0)    # ZeroDivisionError
safe_divide(10, "x")  # TypeError

Result: 5.0
Execution complete.

Error: Cannot divide by zero.
Execution complete.

TypeError: unsupported operand type(s) for /: 'int' and 'str'
Execution complete.



In [128]:
def read_data(filepath):
    file = None
    try:
        file = open(filepath, "r")
        data = file.read()
    except FileNotFoundError:
        print(f"File '{filepath}' not found.")
    except PermissionError:
        print("Permission denied to read the file.")
    except Exception as e:
        print(f"Unexpected error: {e}")
    else:
        print("File read successfully!")
        print(data)
    finally:
        if file:
            file.close()           # always close the file
            print("File closed.")

read_data("data.csv")    # If file exists → else + finally run
read_data("missing.csv") # FileNotFoundError → except + finally run

File 'data.csv' not found.
File 'missing.csv' not found.


### **9. Raising Exceptions using `raise`**

You don't have to wait for Python to raise an exception. You can raise one yourself using the `raise` keyword. This is how you **enforce rules** and **signal problems** to the caller.

```python
raise ExceptionType("Your descriptive error message here")
```

**When should you raise exceptions?**
- Input validation : input doesn't meet requirements
- Business rule violations : an operation isn't allowed
- Impossible states : something that should never happen, did
- API contracts : letting callers know they used your function wrong

**Think of `raise` as a red flag:** you're saying "I detected a problem that I can't solve here, the caller needs to deal with this."


In [132]:
# raise with validation
def set_age(age):
    if not isinstance(age, int):
        raise TypeError(f"Age must be an integer, got {type(age).__name__}")
    if age < 0:
        raise ValueError(f"Age cannot be negative, got {age}")
    if age > 150:
        raise ValueError(f"Age {age} is unrealistically high")
    print(f"Age set to {age}")

# Valid calls
set_age(25)
set_age(0)

# Invalid calls
for bad_input in [-5, 200, "twenty"]:
    try:
        set_age(bad_input)
    except (ValueError, TypeError) as e:
        print(f"Error for {repr(bad_input)}: {e}")

Age set to 25
Age set to 0
Error for -5: Age cannot be negative, got -5
Error for 200: Age 200 is unrealistically high
Error for 'twenty': Age must be an integer, got str


### **10. Custom Exceptions : Building Your Own Error Types**

Python lets you create your own exception classes. This is one of the most powerful and professional features of exception handling.

**Why create custom exceptions?**
- **Clarity:** `InsufficientFundsError` is far more descriptive than a plain `ValueError`
- **Control:** Callers can catch YOUR specific error without catching everything else
- **Context:** You can attach extra information to the exception (like error codes, user IDs)
- **Separation:** Separate library/module errors from Python built-in errors

**How to create one:**
```python
class MyError(Exception):
    pass
```

That's the minimum. You can also add `__init__` to carry extra data.

**Best practices:**
- Always inherit from `Exception` (not `BaseException`)
- Name them with `Error` at the end
- Group related custom exceptions under a common base class


In [143]:
# Basic custom exception
class NegativeValueError(Exception):
    """Raised when a value that must be positive is negative."""
    pass

class EmptyInputError(Exception):
    """Raised when required input is empty or None."""
    pass

def calculate_square_root(number):
    if number is None or number == "":
        raise EmptyInputError("Input cannot be None or empty")
    if number < 0:
        raise NegativeValueError(
            f"Cannot calculate square root of negative number: {number}"
        )
    return number ** 0.5

test_inputs = [16, 0, -9, None]
for val in test_inputs:
    try:
        result = calculate_square_root(val)
        print(f"  sqrt({val}) = {result}")
    except EmptyInputError as e:
        print(f"  EmptyInputError: {e}")
    except NegativeValueError as e:
        print(f"  NegativeValueError: {e}")

  sqrt(16) = 4.0
  sqrt(0) = 0.0
  NegativeValueError: Cannot calculate square root of negative number: -9
  EmptyInputError: Input cannot be None or empty


### **11. `sys.exc_info()` : Inspecting the Current Exception**

`sys.exc_info()` returns a tuple of `(type, value, traceback)` about the exception currently being handled. This is useful when you're writing generic exception handlers that need to inspect what went wrong.

It returns `(None, None, None)` when no exception is being handled.

> **Note:** In most modern code, using `except Exception as e` and inspecting `e` directly is cleaner. `sys.exc_info()` is most useful in frameworks, decorators, and low-level exception tooling.


In [145]:
import sys
import traceback

def risky_operation(value):
    if value == 0:
        raise ZeroDivisionError("Cannot process zero")
    if isinstance(value, str):
        raise TypeError(f"Expected number, got string: {value!r}")
    return 100 / value

def generic_error_reporter(func, *args):
    """Generic wrapper that reports exceptions using sys.exc_info()."""
    try:
        return func(*args)
    except Exception:
        exc_type, exc_value, exc_tb = sys.exc_info()
        
        print(f"  Exception type    : {exc_type.__name__}")
        print(f"  Exception message : {exc_value}")
        print(f"  At line           : {exc_tb.tb_lineno}")
        
        # Get the full traceback as a string
        tb_lines = traceback.format_exception(exc_type, exc_value, exc_tb)
        print(f"  Traceback preview : {tb_lines[-1].strip()}")
        return None

print("Using sys.exc_info() to inspect exceptions:\n")
result = generic_error_reporter(risky_operation, 5)
print(f"  Success: {result}\n")

generic_error_reporter(risky_operation, 0)
print()
generic_error_reporter(risky_operation, "hello")

Using sys.exc_info() to inspect exceptions:

  Success: 20.0

  Exception type    : ZeroDivisionError
  Exception message : Cannot process zero
  At line           : 14
  Traceback preview : ZeroDivisionError: Cannot process zero

  Exception type    : TypeError
  Exception message : Expected number, got string: 'hello'
  At line           : 14
  Traceback preview : TypeError: Expected number, got string: 'hello'


### **Summary: Your Python Exception Handling Cheat Sheet**

### Three Types of Errors
| Type | When | Handleable? |
|------|------|-------------|
| **Syntax Error** | Before running | No |
| **Logical Error** | During running, wrong output | Only by you (tests) |
| **Runtime Error (Exception)** | During running, crash | Yes with `try/except` |

### The Complete try/except/else/finally Structure
```python
try:
    result = risky_operation()     # Only the risky code here!
except SpecificError as e:
    handle_specific(e)             # Runs ONLY for SpecificError
except (ErrorA, ErrorB) as e:
    handle_multiple(e)             # Runs for ErrorA OR ErrorB
else:
    use_result(result)             # Runs ONLY if try had no exception
finally:
    cleanup()                      # ALWAYS runs — no matter what
```

### Custom Exceptions
```python
class MyError(Exception):
    pass

class DetailedError(Exception):
    def __init__(self, message, code=None):
        super().__init__(message)
        self.code = code
```

### Exception Hierarchy (key relationships)
```
Exception
├── ValueError, TypeError, NameError, AttributeError
├── ArithmeticError → ZeroDivisionError, OverflowError
├── LookupError → IndexError, KeyError
└── OSError → FileNotFoundError, PermissionError, TimeoutError
```